# Stochastic Engineering Economics & Dynamic Capital Budgeting

This notebook demonstrates advanced engineering economics workflows:
1. Vectorized after-tax cash flows with IRS MACRS depreciation.
2. Vectorized Monte Carlo valuation (Beta-PERT distributions, VaR, CVaR).
3. Sensitivity analysis and Tornado chart ranking.
4. Economic Service Life (ESL) asset replacement.
5. Exact 0-1 Knapsack capital rationing under budget constraints.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from stochastic_econ.cashflow import CashFlowEngine, MACRSDepreciation
from stochastic_econ.simulation import MonteCarloSimulator, NormalParam, BetaPERTParam, UniformParam
from stochastic_econ.optimization import AssetReplacementOptimizer, CapitalRationingOptimizer, ProjectCandidate
from stochastic_econ.reporting import SensitivityAnalyzer

## 1. Deterministic Project Valuation with 7-Year MACRS

In [ ]:
engine = CashFlowEngine(
    horizon_years=7,
    initial_capex=250_000.0,
    revenues=110_000.0,
    operating_costs=30_000.0,
    tax_rate=0.25,
    working_capital=20_000.0,
    salvage_value=35_000.0,
    depreciation_schedule=MACRSDepreciation(recovery_period=7),
)
val = engine.evaluate(marr=0.12)
print(f"NPV: ${val.npv:,.2f} | IRR: {val.irr * 100:.2f}% | Payback: {val.discounted_payback_years:.2f} yrs")
val.cash_flow_table

## 2. Monte Carlo Risk & Value-at-Risk Simulation

In [ ]:
sim = MonteCarloSimulator(
    horizon_years=7,
    capex_dist=NormalParam(250_000.0, 20_000.0),
    revenue_dist=BetaPERTParam(low=75_000.0, mode=110_000.0, high=155_000.0),
    opex_dist=NormalParam(30_000.0, 4_000.0),
    tax_rate=0.25,
    marr_dist=UniformParam(0.10, 0.14),
    salvage_dist=NormalParam(35_000.0, 8_000.0),
    working_capital_ratio=0.08,
    macrs_period=7,
)
sim_res = sim.run(n_trials=10_000, seed=42)
print(sim_res.summary())

plt.figure(figsize=(9, 4))
plt.hist(sim_res.npv_samples, bins=50, color="steelblue", edgecolor="black", alpha=0.7)
plt.axvline(0, color="red", linestyle="--", label="Breakeven (NPV=0)")
plt.axvline(sim_res.mean_npv, color="green", linestyle="-", label=f"Mean NPV (${sim_res.mean_npv:,.0f})")
plt.title("Monte Carlo NPV Distribution")
plt.xlabel("Net Present Value ($)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. Tornado Sensitivity Analysis

In [ ]:
analyzer = SensitivityAnalyzer(
    horizon_years=7,
    base_capex=250_000.0,
    base_revenue=110_000.0,
    base_opex=30_000.0,
    base_marr=0.12,
    base_tax_rate=0.25,
    base_salvage=35_000.0,
    macrs_period=7,
)
tornado_df = analyzer.compute_tornado(perturbation_pct=0.20)
tornado_df

## 4. Economic Service Life (ESL) Asset Replacement

In [ ]:
repl_opt = AssetReplacementOptimizer(
    initial_cost=120_000.0,
    salvage_schedule=[85_000, 60_000, 42_000, 28_000, 18_000, 10_000],
    operating_costs=[14_000, 18_000, 24_000, 32_000, 42_000, 55_000],
    interest_rate=0.12,
)
esl_res = repl_opt.compute_esl()
print(f"Optimal Economic Service Life: {esl_res.optimal_life_years} years (EUAC = ${esl_res.minimum_euac:,.2f})")
esl_res.annual_cost_table

## 5. Capital Rationing (Exact Knapsack vs Heuristic)

In [ ]:
projects = [
    ProjectCandidate("Project Alpha", capex=80_000, npv=65_000),
    ProjectCandidate("Project Beta", capex=70_000, npv=55_000),
    ProjectCandidate("Project Gamma", capex=50_000, npv=42_000),
    ProjectCandidate("Project Delta", capex=40_000, npv=31_000),
    ProjectCandidate("Project Epsilon", capex=30_000, npv=22_000),
]
budget_opt = CapitalRationingOptimizer(projects, total_budget=140_000.0)
exact_res = budget_opt.solve_exact_knapsack()
pi_res = budget_opt.solve_profitability_index_heuristic()

print(f"Exact 0-1 Knapsack NPV: ${exact_res.total_npv_achieved:,.2f} | Selected: {exact_res.selected_projects}")
print(f"Greedy PI Heuristic NPV: ${pi_res.total_npv_achieved:,.2f} | Selected: {pi_res.selected_projects}")